
# Generative Reservoir Computing (GRC): 2D Reservoir, Sliced Marginals, and Moment Matching (JAX)

This notebook builds a simple **pedagogical** example of:
1. Sampling a *complicated* 2D **reservoir** distribution via a nonlinear warp of a Gaussian mixture.
2. Computing and visualizing **sliced marginals** along different directions (unit vectors on the plane).
3. Performing **moment matching** by learning a cubic polynomial that maps a chosen 1D projection of the reservoir to match a **target** distribution's first four central moments.

> Theory background and notation align with your write-up on moment/feature matching readouts from a high-dimensional reservoir.

### What you'll get
- Clean figures to drop into a proposal: reservoir scatter, multiple 1D marginals, and **before/after** density comparisons for moment matching.
- All core computations are done with **JAX** (arrays + autodiff) and plain **matplotlib** for plotting (no seaborn, no styling).

> Tip: If you don't have GPU, this runs fine on CPU for the sizes used here.


In [ ]:

# JAX + plotting
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

# --- Lightweight 1D Gaussian KDE (no external deps) ---
def kde_1d(samples, grid, bandwidth=None):
    x = jnp.asarray(samples).reshape(-1, 1)
    g = jnp.asarray(grid).reshape(1, -1)
    n = x.shape[0]
    if bandwidth is None:
        std = jnp.std(samples)
        bw = 1.06 * std * (n ** (-1/5))
        bw = jnp.maximum(bw, 1e-6)
    else:
        bw = bandwidth
    z = (g - x) / bw
    phi = jnp.exp(-0.5 * z**2) / jnp.sqrt(2 * jnp.pi)
    dens = jnp.mean(phi, axis=0) / bw
    return dens


In [ ]:

# -----------------------------
# 1) Build a "complicated" 2D reservoir distribution p_R(x)
#    via nonlinear warp of a Gaussian mixture
# -----------------------------
key = jax.random.PRNGKey(0)

def sample_reservoir(key, n=8000):
    keys = jax.random.split(key, 3)
    mean1 = jnp.array([0.0, 0.0])
    mean2 = jnp.array([2.5, -1.2])
    cov1 = jnp.array([[1.0, 0.8],[0.8, 1.5]])
    cov2 = jnp.array([[0.7, -0.4],[-0.4, 1.2]])
    L1 = jnp.linalg.cholesky(cov1)
    L2 = jnp.linalg.cholesky(cov2)
    z1 = jax.random.normal(keys[0], (n//2, 2)) @ L1.T + mean1
    z2 = jax.random.normal(keys[1], (n - n//2, 2)) @ L2.T + mean2
    z = jnp.vstack([z1, z2])
    # non-linear warp
    def warp(v):
        x, y = v[..., 0], v[..., 1]
        r2 = x**2 + y**2
        x_new = x * jnp.cos(0.25 * r2) - y * jnp.sin(0.25 * r2)
        y_new = x * jnp.sin(0.25 * r2) + y * jnp.cos(0.25 * r2)
        y_new = y_new + 0.75 * jnp.tanh(0.6 * x_new)
        return jnp.stack([x_new, y_new], axis=-1)
    return warp(z)

reservoir = sample_reservoir(key, n=8000)

plt.figure(figsize=(5,5))
plt.scatter(np.array(reservoir[:,0]), np.array(reservoir[:,1]), s=2, alpha=0.4)
plt.title("Reservoir samples in 2D"); plt.xlabel("x1"); plt.ylabel("x2"); plt.tight_layout(); plt.show()


In [ ]:

# -----------------------------
# 2) Sliced marginals: project along different directions u(theta)
# -----------------------------
def unit_vec(theta):
    return jnp.array([jnp.cos(theta), jnp.sin(theta)])

thetas = jnp.linspace(0.0, jnp.pi, 5)  # a few directions
dirs = jnp.stack([unit_vec(t) for t in thetas], axis=0)
projections = [reservoir @ d for d in dirs]

for t, s in zip(np.array(thetas), projections):
    plt.figure(figsize=(5,3))
    grid = jnp.linspace(jnp.min(s)-0.5, jnp.max(s)+0.5, 300)
    dens = kde_1d(s, grid)
    plt.plot(np.array(grid), np.array(dens))
    plt.title(f"Marginal along direction θ={float(t):.2f} rad")
    plt.xlabel("projection value"); plt.ylabel("density"); plt.tight_layout(); plt.show()


In [ ]:

# -----------------------------
# 3) Define a 1D target and do moment matching with a cubic polynomial
# -----------------------------
def sample_target(key, n=8000):
    weights = jnp.array([0.5, 0.3, 0.2])
    means = jnp.array([-2.0, 0.5, 2.5])
    stds  = jnp.array([0.6, 0.5, 0.7])
    cats = jax.random.categorical(key, jnp.log(weights), shape=(n,))
    eps = jax.random.normal(key, (n,))
    return means[cats] + stds[cats] * eps

target_samples = sample_target(jax.random.PRNGKey(123), n=8000)

def kurtosis(x):
    x = x - jnp.mean(x)
    m2 = jnp.mean(x**2) + 1e-12
    m4 = jnp.mean(x**4)
    return m4 / (m2**2) - 3.0

k_stats = [kurtosis(p) for p in projections]
best_idx = int(jnp.argmax(jnp.abs(jnp.asarray(k_stats))))
s_res = projections[best_idx]

def central_moments(x):
    mu = jnp.mean(x)
    xc = x - mu
    m2 = jnp.mean(xc**2)
    m3 = jnp.mean(xc**3)
    m4 = jnp.mean(xc**4)
    return mu, m2, m3, m4

mu_t, m2_t, m3_t, m4_t = central_moments(target_samples)

def poly_transform(c, s):
    c0, c1, c2, c3 = c
    return c0 + c1*s + c2*(s**2) + c3*(s**3)

def loss_moments(c, s):
    y = poly_transform(c, s)
    mu, m2, m3, m4 = central_moments(y)
    return (mu - mu_t)**2 + (m2 - m2_t)**2 + 1e-2*(m3 - m3_t)**2 + 1e-3*(m4 - m4_t)**2

grad_loss = jax.grad(loss_moments)

def optimize_moments(s, steps=1200, lr=5e-3):
    c = jnp.array([0.0, 1.0, 0.0, 0.0])  # identity start
    for _ in range(steps):
        c = c - lr * grad_loss(c, s)
    return c

c_opt = optimize_moments(s_res, steps=1200, lr=5e-3)
y_matched = poly_transform(c_opt, s_res)

# --- Figures: before/after ---
plt.figure(figsize=(5,3))
grid_c = jnp.linspace(jnp.minimum(jnp.min(s_res), jnp.min(target_samples))-1.0,
                      jnp.maximum(jnp.max(s_res), jnp.max(target_samples))+1.0, 400)
dens_res = kde_1d(s_res, grid_c)
dens_tar = kde_1d(target_samples, grid_c)
plt.plot(np.array(grid_c), np.array(dens_res), label="Reservoir projection (before)")
plt.plot(np.array(grid_c), np.array(dens_tar), label="Target distribution")
plt.title("Before moment matching"); plt.xlabel("value"); plt.ylabel("density"); plt.legend(); plt.tight_layout(); plt.show()

plt.figure(figsize=(5,3))
dens_match = kde_1d(y_matched, grid_c)
plt.plot(np.array(grid_c), np.array(dens_match), label="Transformed reservoir (after)")
plt.plot(np.array(grid_c), np.array(dens_tar), label="Target distribution")
plt.title("After polynomial moment matching"); plt.xlabel("value"); plt.ylabel("density"); plt.legend(); plt.tight_layout(); plt.show()

print("Optimized coefficients c* =", c_opt)



---

### Exporting figures
- Each figure is shown inline; you can also save them (e.g., `plt.savefig("figure.png", dpi=300)`).
- For proposals: consider exporting the 2D scatter and a **3-panel** strip of marginals, plus before/after curves.

### Notes
- You can replace the target mixture with *any* 1D dataset, or change to sliced-cumulant/MMD objectives per your theory doc.
- Swap cubic for a richer readout (e.g., polynomial features or a small MLP) if you want flexibility beyond moments.
